# 02 - Conversational Chatbot: Multi-Turn Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/02-conversational-chatbot.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Manage a growing message history for multi-turn conversations
2. Implement sliding window memory to bound context size
3. Count tokens to stay within context window limits
4. Summarize older conversation history to preserve key context
5. Persist and restore chat sessions using JSON
6. Build a simple interactive chat loop

---

**Architecture Pattern:** A stateful chatbot that maintains conversation context across turns. The key challenge is managing the messages array as it grows -- balancing context quality against token cost and context window limits.

## Setup

Install the required packages.

In [ ]:
!pip install openai tiktoken --quiet

## Configuration

In [ ]:
import os
import json
import time
import tempfile
from datetime import datetime
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("To use the real API, set your key: export OPENAI_API_KEY='sk-...'")
else:
    print("API key found. Using live OpenAI API.")

client = OpenAI(api_key=api_key if api_key else "sk-mock-key")
MODEL = "gpt-4o-mini"

# Mock response lookup for offline mode
MOCK_RESPONSES = {
    "default": "That's a great question. Based on our conversation so far, I'd say the key point is to focus on practical understanding before diving into theory.",
    "python": "Python is a versatile, high-level programming language known for its readability. It's widely used in data science, web development, and AI/ML applications.",
    "machine learning": "Machine learning is a branch of AI where models learn patterns from data rather than being explicitly programmed. Common approaches include supervised, unsupervised, and reinforcement learning.",
    "deep learning": "Deep learning uses neural networks with many layers to learn hierarchical representations. It excels at tasks like image recognition, natural language processing, and speech synthesis.",
    "transformer": "Transformers use self-attention mechanisms to process sequences in parallel. They are the foundation of modern LLMs like GPT-4 and form the basis of most state-of-the-art NLP models.",
    "summary": "The user has been exploring AI and ML concepts, starting with Python basics, progressing through machine learning fundamentals, and diving into deep learning architectures and transformers."
}

def get_mock_reply(user_msg):
    """Return a contextually relevant mock response."""
    msg_lower = user_msg.lower()
    for key, response in MOCK_RESPONSES.items():
        if key in msg_lower:
            return response
    return MOCK_RESPONSES["default"]

---
## 1. Message History Management

The OpenAI Chat API is stateless -- it has no memory between calls. To create a multi-turn conversation, **you** must maintain the messages array and send the full history with each request. The model then sees the entire conversation and can respond in context.

In [ ]:
class SimpleChat:
    """Basic multi-turn chat that sends the full message history each time."""

    def __init__(self, system_prompt="You are a helpful AI assistant."):
        self.system_prompt = system_prompt
        self.messages = []

    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})

        api_messages = [{"role": "system", "content": self.system_prompt}] + self.messages

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=api_messages,
                max_tokens=200
            )
            reply = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            reply = get_mock_reply(user_message)
            print("Using mock response for demonstration.")

        self.messages.append({"role": "assistant", "content": reply})
        return reply


# Demo: watch the messages array grow
chat = SimpleChat(system_prompt="You are a concise AI tutor. Keep answers to 2 sentences.")

questions = [
    "What is Python?",
    "How is it used in machine learning?",
    "What about deep learning specifically?",
]

for q in questions:
    reply = chat.send(q)
    print(f"User: {q}")
    print(f"Bot:  {reply}")
    print(f"      [Messages in history: {len(chat.messages)}]")
    print()

In [ ]:
# Inspect the messages array structure
print("=== Full Messages Array ===")
for i, msg in enumerate(chat.messages):
    role = msg['role'].upper()
    content_preview = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
    print(f"  [{i}] {role}: {content_preview}")

print(f"\nTotal messages: {len(chat.messages)}")
total_chars = sum(len(m['content']) for m in chat.messages)
print(f"Total characters: {total_chars}")
print(f"Estimated tokens: ~{total_chars // 4}")

---
## 2. Sliding Window Memory

As conversations grow, sending the full history becomes expensive and eventually exceeds the context window. A **sliding window** keeps only the most recent N message pairs, dropping older ones.

**Trade-off:** Bounded cost, but the model loses early context (e.g., user name stated at turn 1).

In [ ]:
class WindowChat:
    """Chat with sliding window memory -- keeps the last N message pairs."""

    def __init__(self, window_size=3, system_prompt="You are a helpful AI assistant."):
        self.system_prompt = system_prompt
        self.window_size = window_size
        self.full_history = []     # Everything (for logging)
        self.active_window = []    # What the LLM actually sees

    def send(self, user_message):
        self.full_history.append({"role": "user", "content": user_message})
        self.active_window.append({"role": "user", "content": user_message})

        # Trim: keep last N pairs (2 * window_size messages)
        max_messages = self.window_size * 2
        if len(self.active_window) > max_messages:
            self.active_window = self.active_window[-max_messages:]

        api_messages = [{"role": "system", "content": self.system_prompt}] + self.active_window

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=api_messages,
                max_tokens=200
            )
            reply = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            reply = get_mock_reply(user_message)
            print("Using mock response for demonstration.")

        self.full_history.append({"role": "assistant", "content": reply})
        self.active_window.append({"role": "assistant", "content": reply})
        return reply


# Demo with window_size=2 (keeps last 2 pairs = 4 messages)
chat = WindowChat(window_size=2, system_prompt="You are a concise AI tutor.")

conversation = [
    "My name is Alice. What is Python?",
    "How is it used in machine learning?",
    "What about deep learning?",
    "Explain transformers.",
    "Do you remember my name?",  # This should be forgotten!
]

for q in conversation:
    reply = chat.send(q)
    print(f"User: {q}")
    print(f"Bot:  {reply}")
    print(f"      [Full history: {len(chat.full_history)} msgs | "
          f"Window: {len(chat.active_window)} msgs (max {chat.window_size * 2})]")
    print()

---
## 3. Token Counting for Context Window

Instead of counting messages, a more precise approach counts **tokens** using the `tiktoken` library. This lets you set a token budget and trim messages when the budget is exceeded.

In [ ]:
try:
    import tiktoken
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    TIKTOKEN_AVAILABLE = True
    print("tiktoken loaded successfully.")
except Exception:
    TIKTOKEN_AVAILABLE = False
    print("tiktoken not available. Using character-based estimation.")


def count_tokens(text):
    """Count tokens in a string. Falls back to char/4 estimate."""
    if TIKTOKEN_AVAILABLE:
        return len(encoding.encode(text))
    return len(text) // 4


def count_messages_tokens(messages):
    """Count total tokens across all messages (including role overhead)."""
    total = 0
    for msg in messages:
        total += 4  # Overhead per message (role, separators)
        total += count_tokens(msg["content"])
    total += 2  # Reply priming
    return total


# Demo: token counting
sample_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a subset of artificial intelligence where systems learn patterns from data rather than being explicitly programmed."},
    {"role": "user", "content": "Can you give me an example?"},
]

print(f"\n{'Message':<50} {'Tokens':>8}")
print("-" * 60)
for msg in sample_messages:
    tokens = count_tokens(msg['content']) + 4
    preview = f"[{msg['role']}] {msg['content'][:35]}..."
    print(f"{preview:<50} {tokens:>8}")

total = count_messages_tokens(sample_messages)
print(f"{'TOTAL':<50} {total:>8}")
print(f"\nContext window remaining: {128000 - total:,} tokens")

In [ ]:
class TokenBudgetChat:
    """Chat that trims messages based on a token budget, not message count."""

    def __init__(self, token_budget=1000, system_prompt="You are a helpful assistant."):
        self.token_budget = token_budget
        self.system_prompt = system_prompt
        self.messages = []

    def _trim_to_budget(self):
        """Remove oldest messages until we are within token budget."""
        system_msg = [{"role": "system", "content": self.system_prompt}]
        while self.messages:
            total = count_messages_tokens(system_msg + self.messages)
            if total <= self.token_budget:
                break
            # Remove oldest pair (user + assistant)
            removed = self.messages.pop(0)
            if self.messages and self.messages[0]["role"] == "assistant":
                self.messages.pop(0)

    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        self._trim_to_budget()

        api_messages = [{"role": "system", "content": self.system_prompt}] + self.messages
        current_tokens = count_messages_tokens(api_messages)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=api_messages,
                max_tokens=150
            )
            reply = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            reply = get_mock_reply(user_message)
            print("Using mock response for demonstration.")

        self.messages.append({"role": "assistant", "content": reply})
        return reply, current_tokens


# Demo with a tight token budget
chat = TokenBudgetChat(token_budget=300, system_prompt="You are a concise tutor.")

for q in ["What is Python?", "How is it used in ML?", "What is deep learning?", "Explain attention."]:
    reply, tokens = chat.send(q)
    print(f"User: {q}")
    print(f"Bot:  {reply[:100]}...")
    print(f"      [Context tokens: {tokens}/{chat.token_budget} | Messages: {len(chat.messages)}]")
    print()

---
## 4. Conversation Summarization

Instead of simply dropping old messages, we can **summarize** them. This preserves key context (user preferences, topics discussed) in a compact form. The summary is prepended to the active window.

In [ ]:
class SummaryChat:
    """Chat that summarizes older messages to preserve context."""

    def __init__(self, summarize_after=3, system_prompt="You are a helpful assistant."):
        self.system_prompt = system_prompt
        self.summarize_after = summarize_after  # Pairs before summarization
        self.messages = []
        self.summary = ""
        self.summary_count = 0

    def _summarize_old_messages(self):
        """Compress older messages into a running summary."""
        keep_recent = 4  # Keep last 2 pairs
        old_messages = self.messages[:-keep_recent]
        recent_messages = self.messages[-keep_recent:]

        old_text = "\n".join(
            f"{m['role'].title()}: {m['content']}" for m in old_messages
        )

        summary_prompt = (
            f"Previous summary: {self.summary}\n\n"
            f"New conversation to incorporate:\n{old_text}\n\n"
            "Write a concise 2-3 sentence summary of the full conversation so far. "
            "Capture key topics discussed, user preferences, and important facts."
        )

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": summary_prompt}],
                max_tokens=100,
                temperature=0.0
            )
            self.summary = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            topics = [m['content'][:40] for m in old_messages if m['role'] == 'user']
            prev = f"Previously: {self.summary} " if self.summary else ""
            self.summary = (
                f"{prev}The user asked about: {'; '.join(topics)}. "
                "The assistant provided concise explanations for each topic."
            )
            print("Using mock response for demonstration.")

        self.messages = recent_messages
        self.summary_count += 1

    def send(self, user_message):
        self.messages.append({"role": "user", "content": user_message})

        # Summarize when history exceeds threshold
        if len(self.messages) > self.summarize_after * 2:
            print("      [Summarizing older messages...]")
            self._summarize_old_messages()

        # Build context with summary prefix
        api_messages = [{"role": "system", "content": self.system_prompt}]
        if self.summary:
            api_messages.append(
                {"role": "system", "content": f"Conversation summary so far: {self.summary}"}
            )
        api_messages.extend(self.messages)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=api_messages,
                max_tokens=200
            )
            reply = response.choices[0].message.content
        except Exception as e:
            print(f"API call failed: {e}")
            reply = get_mock_reply(user_message)
            print("Using mock response for demonstration.")

        self.messages.append({"role": "assistant", "content": reply})
        return reply


# Demo
chat = SummaryChat(summarize_after=2, system_prompt="You are a concise AI tutor.")

conversation = [
    "My name is Alice. What is Python?",
    "How is it used in machine learning?",
    "What about deep learning?",
    "Explain transformers.",
    "Do you remember my name from the start?",
]

for q in conversation:
    print(f"User: {q}")
    reply = chat.send(q)
    print(f"Bot:  {reply}")
    print(f"      [Active msgs: {len(chat.messages)} | Summaries done: {chat.summary_count}]")
    if chat.summary:
        print(f"      [Summary: {chat.summary[:100]}...]")
    print()

---
## 5. Session Persistence (JSON)

For production apps, conversations must survive server restarts. We persist sessions to JSON files. The same pattern applies to databases (PostgreSQL, DynamoDB, Redis).

In [ ]:
class ChatSession:
    """A persistable chat session."""

    def __init__(self, session_id=None, system_prompt="You are a helpful assistant."):
        self.session_id = session_id or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.system_prompt = system_prompt
        self.messages = []
        self.created_at = datetime.now().isoformat()
        self.metadata = {}

    def add_message(self, role, content):
        self.messages.append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        })

    def to_dict(self):
        return {
            "session_id": self.session_id,
            "system_prompt": self.system_prompt,
            "messages": self.messages,
            "created_at": self.created_at,
            "metadata": self.metadata
        }

    def save(self, filepath):
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)
        print(f"Session saved to {filepath}")

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'r') as f:
            data = json.load(f)
        session = cls(
            session_id=data["session_id"],
            system_prompt=data["system_prompt"]
        )
        session.messages = data["messages"]
        session.created_at = data["created_at"]
        session.metadata = data.get("metadata", {})
        print(f"Session loaded from {filepath} ({len(session.messages)} messages)")
        return session

    def get_api_messages(self):
        """Return messages in OpenAI API format (no timestamps)."""
        return [{"role": m["role"], "content": m["content"]} for m in self.messages]


# Demo: create, populate, save, then reload
session = ChatSession(session_id="demo_001", system_prompt="You are a Python tutor.")
session.metadata["user"] = "alice"

# Simulate a conversation
exchanges = [
    ("user", "What are list comprehensions?"),
    ("assistant", "List comprehensions are a concise way to create lists in Python. They follow the pattern: [expression for item in iterable if condition]."),
    ("user", "Give me an example."),
    ("assistant", "Here's an example: squares = [x**2 for x in range(10)] creates a list of squares from 0 to 81."),
]

for role, content in exchanges:
    session.add_message(role, content)

# Save to temp file
save_path = os.path.join(tempfile.gettempdir(), "chat_session_demo.json")
session.save(save_path)

# Reload from file
restored = ChatSession.load(save_path)
print(f"\nRestored session: {restored.session_id}")
print(f"User: {restored.metadata.get('user', 'unknown')}")
print(f"Messages: {len(restored.messages)}")
for m in restored.messages:
    print(f"  [{m['role']}] {m['content'][:60]}...")

In [ ]:
# Show the saved JSON structure
with open(save_path, 'r') as f:
    saved_data = json.load(f)

print("=== Saved JSON Structure ===")
print(json.dumps(saved_data, indent=2)[:1000])
if len(json.dumps(saved_data)) > 1000:
    print("... (truncated)")

---
## 6. Building a Simple Chat Loop

Putting it all together: a chat loop that uses window memory, token counting, and session persistence. This is the pattern used in production chatbot backends.

In [ ]:
class ProductionChat:
    """Production-style chat with window memory, token tracking, and persistence."""

    def __init__(self, session_id=None, window_size=5,
                 system_prompt="You are a helpful assistant."):
        self.session = ChatSession(session_id=session_id, system_prompt=system_prompt)
        self.window_size = window_size
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def chat(self, user_message):
        self.session.add_message("user", user_message)

        # Get windowed messages for the API call
        all_msgs = self.session.get_api_messages()
        max_msgs = self.window_size * 2
        windowed = all_msgs[-max_msgs:] if len(all_msgs) > max_msgs else all_msgs

        api_messages = [{"role": "system", "content": self.session.system_prompt}] + windowed
        context_tokens = count_messages_tokens(api_messages)

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=api_messages,
                max_tokens=200
            )
            reply = response.choices[0].message.content
            self.total_prompt_tokens += response.usage.prompt_tokens
            self.total_completion_tokens += response.usage.completion_tokens
        except Exception as e:
            print(f"API call failed: {e}")
            reply = get_mock_reply(user_message)
            self.total_prompt_tokens += context_tokens
            self.total_completion_tokens += count_tokens(reply)
            print("Using mock response for demonstration.")

        self.session.add_message("assistant", reply)

        return {
            "reply": reply,
            "context_messages": len(windowed),
            "total_messages": len(self.session.messages),
            "context_tokens": context_tokens,
            "cumulative_prompt_tokens": self.total_prompt_tokens,
            "cumulative_completion_tokens": self.total_completion_tokens
        }

    def save(self, filepath):
        self.session.metadata["total_prompt_tokens"] = self.total_prompt_tokens
        self.session.metadata["total_completion_tokens"] = self.total_completion_tokens
        self.session.save(filepath)


# Simulate a multi-turn conversation
app = ProductionChat(
    session_id="prod_demo_001",
    window_size=3,
    system_prompt="You are a concise coding tutor. Keep responses under 3 sentences."
)

demo_conversation = [
    "What is a REST API?",
    "How do I make a GET request in Python?",
    "What about POST requests?",
    "How do I handle errors?",
    "What is authentication?",
]

print("=== Production Chat Demo ===")
print(f"Window size: {app.window_size} pairs\n")

for q in demo_conversation:
    result = app.chat(q)
    print(f"User: {q}")
    print(f"Bot:  {result['reply'][:120]}...")
    print(f"      [Context: {result['context_messages']} msgs / {result['context_tokens']} tokens | "
          f"Total: {result['total_messages']} msgs | "
          f"Cumulative: {result['cumulative_prompt_tokens']}+{result['cumulative_completion_tokens']} tokens]")
    print()

# Save the session
save_path = os.path.join(tempfile.gettempdir(), "prod_chat_demo.json")
app.save(save_path)

---
## Summary

This notebook covered multi-turn conversation management:

| Strategy | Token Growth | Context Quality | Best For |
|---|---|---|---|
| Full History | Linear O(n) | Perfect recall | Short conversations |
| Sliding Window | Constant O(1) | Recent only | Support chats, Q&A |
| Token Budget | Bounded | Recent, precise | Cost-sensitive apps |
| Summary Memory | Slow growth | Compressed overview | Long conversations |

**Key takeaways:**
- The OpenAI API is stateless -- you own the conversation history
- Token counting (via `tiktoken`) is more precise than message counting
- Summarization preserves context at the cost of an extra API call
- Session persistence enables conversation continuity across restarts

**Next:** In notebook 03, we add external knowledge retrieval (RAG) so the chatbot can answer questions about your own documents.